# Structural Dynamics with KoopSim

## Euler-Bernoulli Beam Vibration Analysis

The **Euler-Bernoulli beam** is a fundamental model in structural dynamics. A cantilever beam (fixed at one end, free at the other) vibrates in its natural modes, each with a characteristic frequency.

KoopSim's modal representation uses:
$$\ddot{\eta}_i = -\omega_i^2 \, \eta_i$$

where $\eta_i$ are modal coordinates and $\omega_i$ are the natural frequencies derived from the cantilever eigenvalue problem.

This is a **linear** system, so the Koopman operator should achieve near-perfect recovery. We also demonstrate the `SpringMassDamper` system as a simpler warm-up example.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from koopsim import KoopSim
from koopsim.systems import EulerBernoulliBeam, SpringMassDamper
from koopsim.utils.visualization import (
    plot_trajectory_comparison,
    plot_eigenspectrum,
)

## Part A: Spring-Mass-Damper Chain (Warm-up)

A 3-mass chain connected by springs and dampers. This is a damped linear system with 6 state dimensions (3 positions + 3 velocities).

In [ ]:
smd = SpringMassDamper(n_masses=3, k=2.0, c=0.1, m=1.0)
print(f"System: {smd.name}, state dimension: {smd.state_dim}")

# Generate training data
x0_smd = np.array([1.0, 0.0, -0.5, 0.0, 0.0, 0.0])  # displace mass 1 and 3
dt_smd = 0.01
X_smd, Y_smd = smd.generate_snapshots(
    x0_smd, dt=dt_smd, n_steps=300, n_trajectories=10
)
print(f"Training: {X_smd.shape[0]} snapshot pairs")

In [ ]:
# Train with polynomial dictionary (degree 2 suffices for near-linear system)
sim_smd = KoopSim(method="edmd", poly_degree=2)
sim_smd.fit(X_smd, Y_smd, dt=dt_smd)

# Validate
X_val, Y_val = smd.generate_snapshots(
    np.array([0.5, -0.3, 0.2, 0.0, 0.1, 0.0]),
    dt=dt_smd, n_steps=100, n_trajectories=5
)
rmse = sim_smd.validate(X_val, Y_val, metric="rmse")
print(f"Spring-Mass-Damper one-step RMSE: {rmse:.6e}")

In [ ]:
# Predict and compare
x0_test_smd = np.array([0.8, -0.2, 0.4, 0.0, 0.0, 0.0])
times_smd = np.arange(501) * dt_smd
traj_true_smd = smd.generate_trajectory(x0_test_smd, dt=dt_smd, n_steps=500)
traj_pred_smd = sim_smd.predict_trajectory(x0_test_smd, times_smd)

fig = plot_trajectory_comparison(
    times_smd, traj_true_smd, traj_pred_smd,
    labels=["q1", "q2", "q3", "v1", "v2", "v3"]
)
fig.suptitle("Spring-Mass-Damper: True vs Koopman", y=1.01)
plt.show()

## Part B: Euler-Bernoulli Beam

We model a cantilever beam with 5 finite elements (10 state dimensions: 5 modal displacements + 5 modal velocities). The beam parameters correspond to a steel beam:
- $E = 200$ GPa (Young's modulus)
- $I = 10^{-6}$ m$^4$ (second moment of area)
- $\rho = 7800$ kg/m$^3$ (density)
- $L = 1$ m (length)

In [ ]:
beam = EulerBernoulliBeam(n_elements=5)
print(f"System: {beam.name}, state dimension: {beam.state_dim}")

# Initial condition: excite first two modes
x0_beam = np.zeros(beam.state_dim)
x0_beam[0] = 1.0   # first mode displacement
x0_beam[1] = 0.3   # second mode displacement

# Use a small dt to resolve the highest natural frequency
dt_beam = 1e-5
X_beam, Y_beam = beam.generate_snapshots(
    x0_beam, dt=dt_beam, n_steps=200, n_trajectories=10
)
print(f"Training: {X_beam.shape[0]} snapshot pairs")

In [ ]:
# Train EDMD (identity dictionary is enough for this linear system)
sim_beam = KoopSim(method="edmd")
sim_beam.fit(X_beam, Y_beam, dt=dt_beam)
print(sim_beam)

## Predict Beam Vibration

In [ ]:
x0_test_beam = np.zeros(beam.state_dim)
x0_test_beam[0] = 0.8
x0_test_beam[1] = 0.5
x0_test_beam[2] = 0.1

n_pred = 400
times_beam = np.arange(n_pred + 1) * dt_beam
traj_true_beam = beam.generate_trajectory(x0_test_beam, dt=dt_beam, n_steps=n_pred)
traj_pred_beam = sim_beam.predict_trajectory(x0_test_beam, times_beam)

# Show only the first 3 modal displacements for clarity
fig = plot_trajectory_comparison(
    times_beam * 1000,  # convert to ms
    traj_true_beam[:, :3],
    traj_pred_beam[:, :3],
    labels=["Mode 1", "Mode 2", "Mode 3"]
)
for ax in fig.axes:
    ax.set_xlabel("Time (ms)")
fig.suptitle("Beam Modal Displacements: True vs Koopman", y=1.01)
plt.show()

In [ ]:
# Validation
error_per_step = np.sqrt(np.mean((traj_true_beam - traj_pred_beam)**2, axis=1))
print(f"Mean trajectory RMSE: {np.mean(error_per_step):.6e}")
print(f"Max trajectory RMSE:  {np.max(error_per_step):.6e}")

## Spectral Analysis: Identify Natural Frequencies

The Koopman eigenspectrum should reveal the beam's natural frequencies. For a cantilever beam, the analytical frequencies are:

$$\omega_n = \left(\frac{\beta_n}{L}\right)^2 \sqrt{\frac{EI}{\rho A}}$$

where $\beta_1 L = 1.8751$, $\beta_2 L = 4.6941$, $\beta_3 L = 7.8548$, ...

In [ ]:
fig = plot_eigenspectrum(sim_beam.model)
plt.show()

In [ ]:
spectral = sim_beam.spectral_analysis()

# Extract positive frequencies (each mode appears as a conjugate pair)
freqs_hz = np.abs(spectral["frequencies"]) / (2 * np.pi)
unique_freqs = np.sort(np.unique(np.round(freqs_hz, 1)))
unique_freqs = unique_freqs[unique_freqs > 0]

print("Koopman-identified frequencies (Hz):")
for i, f in enumerate(unique_freqs[:5]):
    print(f"  Mode {i+1}: {f:.1f} Hz")

# Compare to analytical cantilever frequencies
EI = 2e11 * 1e-6
rhoA = 7800.0 * 1e-4
L = 1.0
beta_L = np.array([1.8751, 4.6941, 7.8548, 10.9955, 14.1372])
omega_analytical = (beta_L / L)**2 * np.sqrt(EI / rhoA)
freq_analytical_hz = omega_analytical / (2 * np.pi)

print("\nAnalytical cantilever frequencies (Hz):")
for i, f in enumerate(freq_analytical_hz):
    print(f"  Mode {i+1}: {f:.1f} Hz")

## Quick Start with `from_system`

In [ ]:
sim_quick = KoopSim.from_system(
    SpringMassDamper(n_masses=2, k=1.0, c=0.2),
    dt=0.01, n_steps=200, n_trajectories=10,
    poly_degree=2,
)
x0 = np.array([1.0, 0.0, 0.0, 0.0])
state_at_5 = sim_quick.predict(x0, t=5.0)
print(f"2-mass SMD state at t=5.0: {state_at_5}")

## Summary

In this notebook we:
1. Analyzed a 3-mass spring-mass-damper chain (damped linear system)
2. Modeled an Euler-Bernoulli cantilever beam (5 modal DOFs)
3. Trained EDMD models and validated with low RMSE
4. Used the Koopman eigenspectrum to **identify natural frequencies** and compared to analytical values

For linear structural systems, the Koopman approach achieves near-exact recovery. The eigenspectrum provides a data-driven alternative to modal analysis.